# Session 12 — EnKF with nonlinear forecasts

Download the notebook with the toolbar. Run the supplied baseline from a fresh kernel before changing settings.
Use [the course Python environment](https://feelpp.github.io/course-rom/course-rom/setup.html). Each practical starts independently of your earlier notebooks.
Read [the accompanying notes](https://feelpp.github.io/course-rom/rom/assimilation/enkf.html) for assumptions and derivations.
The timed tasks below occupy 60 minutes, including the closing comparison; optional extensions are outside that budget.
Website plots come from executing these same cells. Synthetic truth is used to evaluate methods, never as an undeclared estimator input.
## A supplied Lorenz forecast model (10 minutes)

Lorenz-63 is a small nonlinear dynamical benchmark, not a thermal-fin model.
RK4 advances at step 0.01; observations of the first and third variables arrive every ten steps.
This experiment has a deterministic forecast model and additive observation noise. Inflation is an explicit finite-ensemble modeling choice.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
def rhs(v):
    x,y,z=v
    return np.array([10*(y-x),x*(28-z)-y,x*y-(8/3)*z])
def advance(v):
    v=v.copy(); dt=.01
    for _ in range(10):
        a=rhs(v); b=rhs(v+dt*a/2); c=rhs(v+dt*b/2); d=rhs(v+dt*c)
        v=v+dt*(a+2*b+2*c+d)/6
    return v
H=np.array([[1.,0.,0.],[0.,0.,1.]])
steps=80
truth=[np.array([1.,1.,20.])]
for k in range(steps): truth.append(advance(truth[-1]))
truth=np.array(truth)
noise_draw=np.random.default_rng(120).normal(size=(steps,2))
prior=np.array([0.,0.,22.])
def analysis(X,y,H,R,rng):
    Y=H@X
    Xa=X-X.mean(axis=1,keepdims=True); Ya=Y-Y.mean(axis=1,keepdims=True)
    Cxy=Xa@Ya.T/(X.shape[1]-1); Cyy=Ya@Ya.T/(X.shape[1]-1)
    gain=np.linalg.solve(Cyy+R,Cxy.T).T
    perturb=rng.multivariate_normal(np.zeros(len(y)),R,size=X.shape[1]).T
    return X+gain@(y[:,None]+perturb-Y)


## Stochastic analysis and ensemble size (20 minutes)

**Task 1.** Explain the observation perturbations and the factor one less than ensemble size in the covariance.
The filter never reads the truth array: it receives only observations, the prior and the model.


In [ ]:
def run_filter(observations,N=40,seed=121,inflation=1.02,noise_sd=1.):
    rng=np.random.default_rng(seed)
    X=prior[:,None]+rng.normal(0,2,(3,N))
    means=[X.mean(axis=1)]; spreads=[X.std(axis=1,ddof=1)]
    for observed in observations:
        X=advance(X)
        center=X.mean(axis=1,keepdims=True)
        X=center+inflation*(X-center)  # Factor multiplies anomalies, not covariance.
        X=analysis(X,observed,H,noise_sd**2*np.eye(2),rng)
        means.append(X.mean(axis=1)); spreads.append(X.std(axis=1,ddof=1))
    return np.array(means),np.array(spreads)
noise_sd=1.
observations=truth[1:]@H.T+noise_sd*noise_draw
means,spreads=run_filter(observations,noise_sd=noise_sd)
forecast=[prior.copy()]
for k in range(steps): forecast.append(advance(forecast[-1]))
forecast=np.array(forecast)
print('EnKF RMSE:',np.sqrt(np.mean((means[1:]-truth[1:])**2)))
print('Forecast-only RMSE:',np.sqrt(np.mean((forecast[1:]-truth[1:])**2)))
for N in (20,80):
    scores=[]
    for seed in (121,122,123):
        result,_=run_filter(observations,N=N,seed=seed,noise_sd=noise_sd)
        scores.append(np.sqrt(np.mean((result[1:]-truth[1:])**2)))
    print(f'Ensemble {N}: RMSE mean={np.mean(scores):.3f}, SD={np.std(scores):.3f}')


## Trajectories and sensitivity (20 minutes)

**Task 2.** Change noise standard deviation to 2 and regenerate observations and the assumed covariance consistently.
Compare the same ensemble seeds. A larger ensemble does not guarantee smaller error in every finite realization.


In [ ]:
t=.1*np.arange(steps+1)
fig,ax=plt.subplots(figsize=(8,3.5))
ax.plot(t,truth[:,0],label='Truth'); ax.plot(t,means[:,0],label='EnKF mean')
ax.plot(t,forecast[:,0],'--',alpha=.6,label='Forecast only')
ax.fill_between(t,means[:,0]-2*spreads[:,0],means[:,0]+2*spreads[:,0],alpha=.2,label='Mean ± 2 sample SD')
ax.set(xlabel='Time',ylabel='Lorenz first component'); ax.legend(fontsize=8)
fig.tight_layout(); plt.show()


## Checkpoint (10 minutes)

Submit an ensemble-size/noise table and distinguish sample spread from a calibrated confidence interval.
**Task 3.** Explain why multiplying every member by an inflation factor would incorrectly alter the mean.
Optional: remove observation perturbations and investigate the resulting spread; identify what would be needed for a valid deterministic square-root method.
